# Числове дослідження МНС і ПАРТАН-МНС

Блокнот оформлено як серію експериментів: для кожного параметра будується таблиця, графіки та короткий висновок. Окремо досліджується безумовна оптимізація та умовна оптимізація методом зовнішніх штрафних функцій.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "optimization").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import optimization.experiments as experiments
experiments = importlib.reload(experiments)

BASE_PARAMS = experiments.BASE_PARAMS
EXPERIMENTS = experiments.EXPERIMENTS
METHODS = experiments.METHODS
DISPLAY_COLUMN_LABELS = experiments.DISPLAY_COLUMN_LABELS
compare_methods = experiments.compare_methods
compare_penalty_methods = experiments.compare_penalty_methods
compare_penalty_start_points = experiments.compare_penalty_start_points
distance_to_circle_boundary = experiments.distance_to_circle_boundary
format_point = experiments.format_point
old_sympy_check = experiments.old_sympy_check
penalty_experiment_summary = experiments.penalty_experiment_summary
sweep = experiments.sweep

from optimization.functions import F_MIN, FUNCTION_FORMULA, X_MIN, X_START, power_function
from optimization.partan_steepest_descent import partan_steepest_descent
from optimization.penalty import circle_constraint, make_external_penalty_function
import optimization.plots as plots
plots = importlib.reload(plots)
plot_circle_constraint = plots.plot_circle_constraint
plot_final_comparison = plots.plot_final_comparison
plot_metric_by_parameter = plots.plot_metric_by_parameter
plot_penalty_trajectory = plots.plot_penalty_trajectory
plot_trajectory = plots.plot_trajectory
from optimization.steepest_descent import steepest_descent

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (8, 4)

base_params = dict(BASE_PARAMS)
base_params

In [ ]:
COLUMN_LABELS = {
    "task": "задача",
    "method": "метод",
    "parameter_value": "значення параметра",
    "x_final": "кінцева точка",
    "f_final": "кінцеве значення функції",
    "grad_norm_final": "норма градієнта",
    "iterations": "кількість ітерацій",
    "func_calls": "кількість викликів функції",
    "status": "статус",
}
COLUMN_LABELS.update(DISPLAY_COLUMN_LABELS)

PARAMETER_LABELS = {
    "derivative_h": "крок h",
    "gradient_scheme": "схема чисельного диференціювання",
    "line_search_method": "метод одновимірного пошуку",
    "line_search_eps": "точність одновимірного пошуку",
    "sven_alpha": "параметр α методу Свена",
    "stop_criterion": "критерій зупинки",
}

CONCLUSIONS = {
    "derivative_h": "Малий крок дає точніше чисельне наближення похідної, але може збільшувати кількість обчислень або викликати числову нестабільність. Надто великий крок може давати грубе наближення градієнта.",
    "gradient_scheme": "Центральна схема зазвичай дає точніше наближення похідної, але може потребувати більше викликів функції, бо для кожної координати функція обчислюється з двох боків.",
    "line_search_method": "Метод одновимірного пошуку впливає не тільки на точність вибору кроку λ, але й на загальну кількість викликів цільової функції.",
    "line_search_eps": "Занадто груба точність одновимірного пошуку може погіршити кінцеву точність, а занадто мала точність може збільшити кількість викликів функції без суттєвого покращення результату.",
    "sven_alpha": "Параметр Свена впливає на початкову локалізацію інтервалу для одновимірного пошуку. Невдале значення може збільшити кількість обчислень функції.",
    "stop_criterion": "Комбінований критерій враховує не тільки норму градієнта, але й зміну точки або значення функції, тому може бути практичнішим для чисельних методів.",
}


def format_vector(x):
    arr = np.asarray(x, dtype=float).reshape(-1)
    return "[" + ", ".join(f"{value:.8f}" for value in arr) + "]"


def parse_point(value):
    if isinstance(value, str):
        return np.fromstring(value.strip("[]"), sep=",")
    return np.asarray(value, dtype=float).reshape(-1)


def format_display_value(value):
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6e}"
    return value


def show_table(df):
    formatted = df.copy()
    for column in formatted.columns:
        formatted[column] = formatted[column].map(format_display_value)
    display(formatted.rename(columns=COLUMN_LABELS))


def sweep_table(parameter_name):
    frames = []
    for method_name, method_fn in METHODS.items():
        df = sweep(method_fn, parameter_name, EXPERIMENTS[parameter_name], base_params=base_params)
        df.insert(0, "method", method_name)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def show_sweep(parameter_name, title):
    table = sweep_table(parameter_name)
    display(Markdown(f"### {title}"))
    show_table(table)

    log_x = parameter_name in {"derivative_h", "line_search_eps", "sven_alpha"}
    xlabel = PARAMETER_LABELS.get(parameter_name, parameter_name)
    metrics = [
        ("iterations", "кількість ітерацій", False),
        ("func_calls", "кількість викликів функції", False),
        ("f_final", "кінцеве значення функції", True),
        ("grad_norm_final", "норма градієнта", True),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for ax, (metric, ylabel, log_y) in zip(axes.ravel(), metrics):
        plot_metric_by_parameter(
            table,
            parameter_col="parameter_value",
            metric_col=metric,
            title=f"{xlabel} -> {ylabel}",
            xlabel=xlabel,
            ylabel=ylabel,
            log_x=log_x,
            log_y=log_y,
            ax=ax,
        )
    plt.tight_layout()
    plt.show()
    display(Markdown(f"**Висновок.** {CONCLUSIONS[parameter_name]}"))
    return table


def contour_grid(points):
    pts = np.asarray(points, dtype=float)
    x_min = min(float(pts[:, 0].min()), -1.5) - 0.25
    x_max = max(float(pts[:, 0].max()), 1.2) + 0.25
    y_min = min(float(pts[:, 1].min()), -0.5) - 0.25
    y_max = max(float(pts[:, 1].max()), 1.2) + 0.25
    x1 = np.linspace(x_min, x_max, 260)
    x2 = np.linspace(y_min, y_max, 260)
    X, Y = np.meshgrid(x1, x2)
    Z = np.vectorize(lambda a, b: power_function(np.array([a, b], dtype=float)))(X, Y)
    return X, Y, Z


def plot_combined_trajectories(results, title):
    all_points = np.vstack([result["points"] for result in results.values()])
    X, Y, Z = contour_grid(all_points)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.contour(X, Y, Z, levels=28, cmap="viridis", alpha=0.65)
    colors = {"МНС": "tab:blue", "ПАРТАН-МНС": "tab:orange"}
    for method_name, result in results.items():
        points = np.asarray(result["points"], dtype=float)
        ax.plot(points[:, 0], points[:, 1], "o-", linewidth=1.8, markersize=4.5, color=colors[method_name], label=method_name)
        ax.scatter(points[-1, 0], points[-1, 1], s=85, color=colors[method_name], edgecolor="black", linewidth=1.0)
    ax.scatter([X_START[0]], [X_START[1]], marker="s", s=60, color="tab:purple", label="стартова точка")
    ax.scatter([X_MIN[0]], [X_MIN[1]], marker="*", s=120, color="tab:green", label="очікуваний мінімум")
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.set_aspect("equal", adjustable="box")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Початкова функція та стартові дані

In [ ]:
initial_table = pd.DataFrame([
    {"назва": "функція", "значення": FUNCTION_FORMULA},
    {"назва": "x_start", "значення": format_vector(X_START)},
    {"назва": "x_min", "значення": format_vector(X_MIN)},
    {"назва": "f_min", "значення": F_MIN},
    {"назва": "f(x_start)", "значення": power_function(X_START)},
])
display(initial_table)

x1 = np.linspace(-1.5, 1.5, 260)
x2 = np.linspace(-0.5, 1.5, 260)
X, Y = np.meshgrid(x1, x2)
Z = np.vectorize(lambda a, b: power_function(np.array([a, b], dtype=float)))(X, Y)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(X, Y, Z, levels=30, cmap="viridis")
ax.scatter([X_START[0]], [X_START[1]], color="tab:red", label="x_start")
ax.scatter([X_MIN[0]], [X_MIN[1]], color="tab:green", label="x_min")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Лінії рівня цільової функції")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Пояснення чисельних методів

У роботі похідні обчислюються чисельно. Порівнюються пряма, зворотна та центральна схеми. Центральна схема зазвичай точніша, але потребує більше викликів функції.

На кожній ітерації напрямок спуску уточнюється одновимірним пошуком. Спочатку метод Свена знаходить інтервал, після чого мінімум шукається методом золотого перетину або методом ДСК-Пауелла. Точність `line_search_eps` важлива, бо для степеневої функції оптимальний крок може бути дуже малим.

Для штрафної функції використовується комбінований критерій зупинки, бо поблизу межі області градієнт штрафу може поводитися жорстко, хоча точка вже майже не змінюється.

## 4. Безумовна оптимізація

In [ ]:
h_table = show_sweep("derivative_h", "4.1. Вплив кроку чисельного диференціювання h")

In [ ]:
scheme_table = show_sweep("gradient_scheme", "4.2. Вплив схеми чисельного диференціювання")

In [ ]:
line_search_method_table = show_sweep("line_search_method", "4.3. Вплив методу одновимірного пошуку")

In [ ]:
line_search_eps_table = show_sweep("line_search_eps", "4.4. Вплив точності одновимірного пошуку")

In [ ]:
sven_alpha_table = show_sweep("sven_alpha", "4.5. Вплив параметра методу Свена")

In [ ]:
stop_criterion_table = show_sweep("stop_criterion", "4.6. Вплив критерію зупинки")

### 4.7. Порівняння МНС і ПАРТАН-МНС

In [ ]:
comparison_table = compare_methods(base_params)
show_table(comparison_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_final_comparison(comparison_table.rename(columns={"iterations": "iterations_total"}), "iterations_total", "Порівняння за кількістю ітерацій", ax=axes[0])
plot_final_comparison(comparison_table.rename(columns={"func_calls": "func_calls_total"}), "func_calls_total", "Порівняння за кількістю викликів функції", ax=axes[1])
plt.tight_layout()
plt.show()

display(Markdown("**Висновок.** Порівняння показує різницю між методами за кількістю ітерацій, викликів функції та точністю фінальної точки."))

### 4.8. Траєкторії пошуку

In [ ]:
mns_result = steepest_descent(power_function, X_START, **base_params)
partan_result = partan_steepest_descent(power_function, X_START, **base_params)
trajectory_results = {"МНС": mns_result, "ПАРТАН-МНС": partan_result}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_trajectory(power_function, mns_result["points"], "Траєкторія пошуку МНС", cmap="Blues", ax=axes[0])
plot_trajectory(power_function, partan_result["points"], "Траєкторія пошуку ПАРТАН-МНС", cmap="Oranges", ax=axes[1])
plt.tight_layout()
plt.show()

plot_combined_trajectories(trajectory_results, "Порівняння траєкторій МНС і ПАРТАН-МНС")
display(Markdown("**Висновок.** ПАРТАН-МНС може рухатися до мінімуму іншою траєкторією та потребувати менше ітерацій для досягнення близької точки."))

## 5. Умовна оптимізація методом штрафних функцій

### 5.1. Пояснення методу

Метод штрафних функцій дозволяє розв’язувати задачу умовної оптимізації через послідовність задач безумовної оптимізації.

Початкова задача має вигляд:

`min f(x), при g(x) <= 0`.

Для цього будується допоміжна функція:

`F(x, r) = f(x) + r * P(x)`,

де `P(x)` — штраф за порушення обмеження.

У роботі використовується зовнішня штрафна функція:

`P(x) = max(0, g(x))^2`.

Якщо точка знаходиться всередині допустимої області, штраф дорівнює нулю. Якщо точка виходить за межі області, штраф зростає.

Конкретне обмеження:

`g(x) = x1^2 + x2^2 - 1 <= 0`.

### 5.2. Вплив коефіцієнта штрафу r

In [ ]:
R_VALUES = (1, 10, 100, 1000, 10000, 100000)
penalty_tables = compare_penalty_methods(base_params=base_params, r_values=R_VALUES)
penalty_long = pd.concat(
    [table.assign(method=method_name) for method_name, table in penalty_tables.items()],
    ignore_index=True,
)

for method_name, table in penalty_tables.items():
    display(Markdown(f"#### {method_name}"))
    show_table(table)

metrics = [
    ("iterations", "кількість ітерацій", False),
    ("func_calls", "кількість викликів функції", False),
    ("violation", "порушення обмеження", True),
    ("constraint_value", "значення g(x)", True),
    ("f_original", "значення початкової функції", True),
    ("F_penalty", "значення штрафної функції", True),
]
fig, axes = plt.subplots(3, 2, figsize=(13, 11))
for ax, (metric, ylabel, log_y) in zip(axes.ravel(), metrics):
    plot_metric_by_parameter(
        penalty_long,
        parameter_col="r",
        metric_col=metric,
        title=f"r -> {ylabel}",
        xlabel="r",
        ylabel=ylabel,
        log_x=True,
        log_y=log_y,
        ax=ax,
    )
plt.tight_layout()
plt.show()

display(Markdown("**Висновок.** При збільшенні коефіцієнта штрафу r порушення обмеження зменшується, але обчислювальна складність може зростати."))

### 5.3. Порушення обмеження від r

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_metric_by_parameter(
    penalty_long,
    parameter_col="r",
    metric_col="violation",
    title="Порушення обмеження залежно від r",
    xlabel="r",
    ylabel="max(0, x1^2 + x2^2 - 1)",
    log_x=True,
    log_y=True,
    ax=ax,
)
plt.tight_layout()
plt.show()

### 5.4. Відстань до межі від r

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_metric_by_parameter(
    penalty_long,
    parameter_col="r",
    metric_col="distance_to_boundary",
    title="Відстань фінальної точки до межі області",
    xlabel="r",
    ylabel="abs(sqrt(x1^2 + x2^2) - 1)",
    log_x=True,
    log_y=True,
    ax=ax,
)
plt.tight_layout()
plt.show()

display(Markdown("**Висновок.** Цей графік показує, наскільки близько фінальна точка знаходиться до межі допустимої області."))

### 5.5. Координати фінальної точки від r

In [ ]:
coordinate_rows = []
for method_name, table in penalty_tables.items():
    for _, row in table.iterrows():
        point = parse_point(row["x_final"])
        coordinate_rows.append({"method": method_name, "r": row["r"], "x1_final": point[0], "x2_final": point[1]})
coordinate_table = pd.DataFrame(coordinate_rows)

fig, ax = plt.subplots(figsize=(9, 5))
for method_name, group in coordinate_table.groupby("method", sort=False):
    ax.plot(group["r"], group["x1_final"], "o-", linewidth=1.8, label=f"x1, {method_name}")
    ax.plot(group["r"], group["x2_final"], "s--", linewidth=1.6, label=f"x2, {method_name}")
expected = 1 / np.sqrt(2)
ax.axhline(expected, color="black", linestyle=":", linewidth=1.4, label="1 / sqrt(2)")
ax.set_xscale("log")
ax.set_xlabel("r")
ax.set_ylabel("координата фінальної точки")
ax.set_title("Координати фінальної точки залежно від r")
ax.grid(True, which="both", linestyle="--", alpha=0.35)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

display(Markdown("**Висновок.** При збільшенні коефіцієнта штрафу координати фінальної точки наближаються до точки на межі допустимої області."))

### 5.6. Траєкторії штрафного методу

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
for ax, (method_name, table) in zip(axes, penalty_tables.items()):
    plot_table = table.copy()
    plot_table["x_start"] = format_vector(X_START)
    plot_penalty_trajectory(plot_table, f"Траєкторія штрафного методу для {method_name}", ax=ax)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
plot_circle_constraint(ax)
colors = {"МНС": "tab:blue", "ПАРТАН-МНС": "tab:orange"}
for method_name, table in penalty_tables.items():
    points = np.vstack([parse_point(value) for value in table["x_final"]])
    ax.plot(points[:, 0], points[:, 1], "o-", linewidth=1.8, color=colors[method_name], label=method_name)
    ax.scatter(points[-1, 0], points[-1, 1], s=90, color=colors[method_name], edgecolor="black", linewidth=1.0)
ax.scatter([X_START[0]], [X_START[1]], marker="s", s=60, color="tab:purple", label="стартова точка")
ax.scatter([1.0], [1.0], marker="x", s=90, color="tab:red", linewidths=2.2, label="безумовний мінімум")
ax.set_title("Порівняння траєкторій штрафного методу для МНС і ПАРТАН-МНС")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 5.7. Вплив початкової точки

In [ ]:
penalty_start_points = [
    [-1.5, 0.0],
    [1.5, 0.0],
    [0.0, 1.5],
    [-1.0, -1.0],
    [2.0, 2.0],
]
start_point_table = compare_penalty_start_points(
    base_params=base_params,
    start_points=penalty_start_points,
    r_values=(1, 10, 100, 1000, 10000),
)
show_table(start_point_table)

start_plot_table = start_point_table.copy()
start_plot_table["start_label"] = start_plot_table["x_start"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric, ylabel, log_y in [
    (axes[0, 0], "iterations_total", "сумарна кількість ітерацій", False),
    (axes[0, 1], "func_calls_total", "сумарна кількість викликів функції", False),
    (axes[1, 0], "violation", "порушення обмеження", True),
    (axes[1, 1], "distance_to_boundary", "відстань до межі", True),
]:
    plot_metric_by_parameter(
        start_plot_table,
        parameter_col="start_label",
        metric_col=metric,
        title=f"початкова точка -> {ylabel}",
        xlabel="початкова точка",
        ylabel=ylabel,
        log_y=log_y,
        ax=ax,
    )
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, x_start in zip(axes, penalty_start_points[:3]):
    table = experiments.penalty_experiment(
        steepest_descent,
        base_params=base_params,
        r_values=(1, 10, 100, 1000, 10000),
        x_start=x_start,
    )
    table["x_start"] = format_vector(x_start)
    plot_penalty_trajectory(table, f"МНС, x_start={format_vector(x_start)}", ax=ax)
plt.tight_layout()
plt.show()

display(Markdown("**Висновок.** Початкова точка впливає на кількість ітерацій і викликів функції, але при достатньо великому штрафі метод приводить розв’язок до допустимої області."))

### 5.8. Порівняння МНС і ПАРТАН-МНС для штрафної функції

In [ ]:
penalty_summary = pd.DataFrame([
    penalty_experiment_summary(table, method=method_name)
    for method_name, table in penalty_tables.items()
])
show_table(penalty_summary)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric, title in [
    (axes[0, 0], "iterations_total", "Порівняння за сумарною кількістю ітерацій"),
    (axes[0, 1], "func_calls_total", "Порівняння за сумарною кількістю викликів"),
    (axes[1, 0], "violation", "Порівняння за порушенням обмеження"),
    (axes[1, 1], "distance_to_boundary", "Порівняння за відстанню до межі"),
]:
    plot_final_comparison(penalty_summary, metric, title, ax=ax)
plt.tight_layout()
plt.show()

display(Markdown("**Висновок.** Порівняння показує, який метод швидше розв’язує штрафну задачу та який метод дає менше порушення обмеження."))

## 6. Фінальне порівняння

In [ ]:
unconstrained_final = comparison_table.copy()
unconstrained_final["task"] = "безумовна оптимізація"
unconstrained_final["r_final"] = "-"
unconstrained_final["F_penalty"] = "-"
unconstrained_final["constraint_value"] = "-"
unconstrained_final["violation"] = "-"
unconstrained_final["distance_to_boundary"] = "-"
unconstrained_final["iterations_total"] = unconstrained_final["iterations"]
unconstrained_final["func_calls_total"] = unconstrained_final["func_calls"]

penalty_final = penalty_summary.copy()
penalty_final["task"] = "штрафна функція"
penalty_final["f_final"] = penalty_final["f_original"]
penalty_final["grad_norm_final"] = "-"

final_comparison = pd.concat([
    unconstrained_final[["task", "method", "r_final", "x_final", "f_final", "grad_norm_final", "F_penalty", "constraint_value", "violation", "distance_to_boundary", "iterations_total", "func_calls_total", "status"]],
    penalty_final[["task", "method", "r_final", "x_final", "f_final", "grad_norm_final", "F_penalty", "constraint_value", "violation", "distance_to_boundary", "iterations_total", "func_calls_total", "status"]],
], ignore_index=True)

show_table(final_comparison)

## 7. Контроль через старі SymPy-версії

Цей блок потрібен лише для перевірки траєкторій. Старі реалізації використовують аналітичні похідні та точний пошук кроку через SymPy, тому кількість ітерацій обмежена.

In [ ]:
old_check = old_sympy_check(max_iter_mns=10, max_iter_partan=10, eps=1e-6)
current_check = compare_methods(BASE_PARAMS)
current_check["джерело"] = "поточна NumPy-версія"
old_table = old_check["table"].copy()
old_table["джерело"] = "стара SymPy-версія"
show_table(pd.concat([current_check, old_table], ignore_index=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_trajectory(power_function, old_check["mns_points"], "Контрольна траєкторія МНС (SymPy)", cmap="Greens", ax=axes[0])
plot_trajectory(power_function, old_check["partan_points"], "Контрольна траєкторія ПАРТАН-МНС (SymPy)", cmap="Purples", ax=axes[1])
plt.tight_layout()
plt.show()

## 8. Висновки

У роботі було досліджено метод найшвидшого спуску та метод ПАРТАН-МНС для задачі безумовної оптимізації.

Було перевірено вплив кроку чисельного диференціювання, схеми обчислення похідних, методу одновимірного пошуку, точності одновимірного пошуку, параметра методу Свена та критерію зупинки.

Також було реалізовано метод зовнішніх штрафних функцій для задачі умовної оптимізації з обмеженням `x1^2 + x2^2 <= 1`.

Експерименти показали, що при збільшенні коефіцієнта штрафу `r` порушення обмеження зменшується, а фінальна точка наближається до межі допустимої області.

Порівняння МНС і ПАРТАН-МНС дозволило оцінити різницю між методами за кількістю ітерацій, кількістю викликів функції та точністю отриманого результату.